In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from spektral.layers import GCNConv
from spektral.data import Dataset, Graph, Loader
from spektral.transforms import GCNFilter
from spektral.data.loaders import BatchLoader
from pathlib import Path

# --- Config ---
csv_path = Path("/path/to/training/data.csv")  # Your training CSV
target_size = (32, 32)  # Resize spectrograms
channels = 1  # grayscale
input_dim = target_size[0] * target_size[1]
epochs = 50
batch_size = 32

# --- Build grid adjacency matrix ---
def build_grid_adjacency(h, w):
    from scipy.sparse import lil_matrix
    adj = lil_matrix((h * w, h * w))

    for row in range(h):
        for col in range(w):
            idx = row * w + col
            if row > 0: adj[idx, idx - w] = 1  # up
            if row < h - 1: adj[idx, idx + w] = 1  # down
            if col > 0: adj[idx, idx - 1] = 1  # left
            if col < w - 1: adj[idx, idx + 1] = 1  # right

    return adj.tocsr()

adj = build_grid_adjacency(*target_size)

# --- Custom Dataset ---
class SpectrogramGCNDataset(Dataset):
    def __init__(self, csv_file, **kwargs):
        self.df = pd.read_csv(csv_file)
        super().__init__(**kwargs)

    def read(self):
        output = []
        for i, row in self.df.iterrows():
            spec = np.load(row["filepath"]).astype(np.float32)
            spec = tf.image.resize(spec[..., np.newaxis], target_size).numpy()
            x = spec.reshape(-1, 1)  # Node features: (num_nodes, 1)
            y = np.array([row["label"]], dtype=np.float32)
            output.append(Graph(x=x, a=adj, y=y))
        return output

# --- Load Dataset ---
dataset = SpectrogramGCNDataset(csv_path, transforms=GCNFilter())
loader = BatchLoader(dataset, batch_size=batch_size, epochs=epochs, shuffle=True)

# --- Define the Spectral GCN Model ---
class GCNRegression(Model):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(64, activation='relu')
        self.conv2 = GCNConv(128, activation='relu')
        self.conv3 = GCNConv(256, activation='relu')
        self.pool = GlobalAveragePooling1D()
        self.d1 = Dense(128, activation='relu')
        self.d2 = Dense(64, activation='relu')
        self.out = Dense(1, activation='linear')  # Regression

    def call(self, inputs):
        x, a = inputs
        x = self.conv1([x, a])
        x = self.conv2([x, a])
        x = self.conv3([x, a])
        x = self.pool(x)
        x = self.d1(x)
        x = self.d2(x)
        return self.out(x)

# --- Train the Model ---
model = GCNRegression()

# Build the model to initialize weights
dummy_x = tf.random.normal((1, input_dim, 1))  # 1 sample, 1024 nodes, 1 feature
dummy_a = tf.sparse.from_dense(tf.convert_to_tensor(adj.toarray()[np.newaxis]))  # 1 graph
model((dummy_x, dummy_a))  # trigger build

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    for step in range(loader.steps_per_epoch):
        batch = loader.__next__()
        inputs, targets = batch  # Correct unpacking of ((x, a), y)
        loss, mae = model.train_on_batch(inputs, targets)
        print(f"Step {step + 1}/{loader.steps_per_epoch} - loss: {loss:.4f} - mae: {mae:.4f}")

# --- Save the model ---
model.save_weights("gcn_regression_model_weights.h5")
print("✅ GCN model trained and weights saved.")
